# MOMENT 교차검증 (445명 중 3유도 환자, ML과 같은 fold)

사전학습된 시계열 foundation model **MOMENT-1-base**(AutonLab, backbone 고정)로 Holter 30초 구간을 임베딩하고, 그 위에 MIL head 또는 선형 probe를 학습해 4년 내 SCD를 예측합니다. 평가는 **ML과 같은 5-fold × 3회 교차검증**이고, 고른 후보 1개를 test에서 한 번 평가합니다.

- 입력: 3유도(X·Y·Z), 환자당 30초 구간 64개(24시간에 고르게), 200 Hz → 100 Hz. 구간마다 512 샘플 조각 5개를 유도별로 임베딩해 평균. 임베딩은 처음 한 번만 계산해 `Members/mingyu/datas/moment_cache/`에 저장

- 데이터: `basic.yaml`의 445명 split에서 **Z 없는 2유도 환자는 학습·평가 모두 제외** (환자 단위, 증강본 포함). 교차검증 = 개발 원본 338명(SCD 38명), test 85명(SCD 9명)은 마지막에 한 번만
- fold: ML `results/ml_kfold/kfold_logloss_balanced/folds.csv`를 338명으로 제한 (ML only445와 같은 fold, 2유도 환자만 빠짐)
- 학습 fold: 나머지 원본 + 그 환자들의 SCD 증강본. 평가 fold: 원본만
- 후보: MIL head 2개(v1·v2 설정) + 선형 probe(임베딩 평균 → PCA → 로지스틱) 12개
- MIL best epoch: 학습 fold 안에서 원본 20%를 떼어 그 log loss로 고름 (평가 fold는 쓰지 않음)
- 클래스 가중치: 학습 데이터의 SCD 비율로 고정 (MIL `pos_weight`, 로지스틱 `class_weight="balanced"`)
- 선택 기준: OOF **가중 log loss** (ML과 같음), threshold 0.5 고정
- MIL은 epoch별 train/val loss를 `loss_curve/<후보>.csv`(CV)와 `test_loss_curve.csv`(최종 모델)로 저장 (PPT Loss 그래프용)
- 결과: `Members/mingyu/results/moment_cv/<run_tag>/`. 후보별 OOF는 끝날 때마다 저장되고, 다시 실행하면 끝난 후보는 건너뜁니다.

In [ ]:
import itertools
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

repo = Path.cwd().resolve()
while not (repo / "Members/mingyu/model/finetuning/moment_probe.py").is_file():
    if repo.parent == repo:
        raise RuntimeError("저장소 내부에서 노트북을 실행하세요")
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from Members.mingyu.model.config import load_basic_config  # noqa: E402
from Members.mingyu.model.evaluation.core import binary_metrics  # noqa: E402
from Members.mingyu.model.finetuning import moment_probe as mp  # noqa: E402

## 실행 설정

In [ ]:
settings = {
    "model_name": "AutonLab/MOMENT-1-base",
    # 입력 (moment.ipynb와 같아야 같은 임베딩 캐시를 씀)
    "n_windows": 64,
    "decimation": 2,
    "chunks_per_window": 5,
    "embed_batch_size": 256,
    # MIL 학습 (모든 MIL 후보 공통)
    "batch_size": 16,
    "epochs": 200,
    "patience": 30,
    "min_delta": 1e-4,
    "selection_metric": "val_loss",  # best epoch: 학습 fold 안에서 떼어 낸 원본 20%의 가중 log loss
    "inner_val_fraction": 0.2,
    "seed": 42,
}
mil_configs = {
    "MIL_v1": {"hidden_dim": 128, "dropout": 0.3, "learning_rate": 1e-3, "weight_decay": 1e-2},
    "MIL_v2": {"hidden_dim": 64, "dropout": 0.5, "learning_rate": 1e-4, "weight_decay": 0.1},
}
probe_grid = {"pooling": ["mean", "mean_std"], "pca": [16, 64], "C": [0.01, 0.1, 1.0]}
THRESHOLD = 0.5

fold_file = repo / "Members/mingyu/results/ml_kfold/kfold_logloss_balanced/folds.csv"
ml_oof_file = (
    repo
    / "Members/mingyu/results/ml_kfold/kfold_logloss_balanced/LR_MUSIC_SCD_iterative_only445/oof.csv"
)  # 비교용 ML
run_tag = "moment_cv_445_xyz"  # 설정·fold를 바꾸면 새 이름으로 (moment_cv_445 = 2유도 평가 포함한 이전 결과)
final_method = None  # None이면 OOF 가중 log loss 1등
run_final_test = True  # 선택 규칙(가중 log loss 1등)이 미리 정해져 있어 한 번에 실행

basic = load_basic_config(repo / "Members/mingyu/model/basic.yaml")
raw = basic["data"]["raw_ecg"]
settings["sample_rate_hz"] = int(raw["sample_rate_hz"])
settings["segment_seconds"] = int(raw["segment_seconds"])
tag = f"{settings['model_name'].split('/')[-1].lower()}_w{settings['n_windows']}_d{settings['decimation']}_c{settings['chunks_per_window']}"
cache_dir = repo / "Members/mingyu/datas/moment_cache" / tag
settings["cache_dir"] = str(cache_dir)
result_dir = repo / "Members/mingyu/results/moment_cv" / run_tag

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, cache_dir, result_dir, sep="\n")

## 1. 코호트 (445명)

원본 환자 ID(`base_id`)와 증강 여부를 붙입니다. 445명이 아니면 여기서 멈추고, Z 없는 2유도 환자를 뺍니다.

In [ ]:
cohort = mp.load_cohort(
    Path(raw["split_path"]),
    Path(raw["quality_index_path"]),
    Path(raw["wave_directory"]),
    Path(raw["lead_metadata_path"]),
)
cohort["base_id"] = cohort.patient_id.str.split("-").str[0]
cohort["augmented"] = cohort.patient_id.ne(cohort.base_id)
cohort["dev"] = cohort.split.isin(["train", "validation"])
assert cohort.base_id.nunique() == 445, "445명 split이 아닙니다 (basic.yaml 확인)"

# Z 없는 2유도 환자는 학습·평가 모두 제외 (환자 단위: 원본에 Z가 없으면 그 환자의 증강본도 제외)
two_lead = set(cohort.base_id[~cohort.has_z & ~cohort.augmented])
assert cohort.loc[~cohort.has_z, "base_id"].isin(two_lead).all(), (
    "원본은 3유도인데 Z 없는 증강본이 있음"
)
print(f"2유도 제외: 환자 {len(two_lead)}명, 파일 {int(cohort.base_id.isin(two_lead).sum())}개")
cohort = cohort[~cohort.base_id.isin(two_lead)].reset_index(drop=True)
y = cohort.y_true.to_numpy()

dev_ids = cohort.patient_id[cohort.dev & ~cohort.augmented].to_numpy()
test_ids = cohort.patient_id[cohort.split.eq("test") & ~cohort.augmented].to_numpy()
assert len(dev_ids) + len(test_ids) == 445 - len(two_lead) and cohort.has_z.all()
assert cohort.loc[cohort.augmented, "base_id"].isin(dev_ids).all(), "test 환자의 증강본이 있음"

groups = {
    "교차검증 원본": cohort.dev & ~cohort.augmented,
    "증강 파일": cohort.augmented,
    "test": cohort.split.eq("test"),
}
pd.DataFrame(
    {
        name: {
            "files": int(m.sum()),
            "scd": int(y[m].sum()),
            "two_lead": int((~cohort.has_z[m]).sum()),
        }
        for name, m in groups.items()
    }
)

## 2. MOMENT 임베딩

캐시에 없는 파일만 backbone으로 계산합니다 (`moment.ipynb`와 같은 캐시). 모두 있으면 바로 불러옵니다.

In [ ]:
missing = cohort.loc[[not (cache_dir / f"{pid}.npy").is_file() for pid in cohort.patient_id]]
print(f"캐시에 없는 파일: {len(missing)} / {len(cohort)}")
if len(missing):
    backbone = mp.load_moment(settings["model_name"], device)
    mp.extract_embeddings(missing, backbone, settings, device)
    del backbone
    torch.cuda.empty_cache()

features, mask = mp.load_bags(cohort, cache_dir)  # (파일, window, 3유도 x 768), cohort와 같은 순서
weights = mask.unsqueeze(-1).float()
counts = weights.sum(1)
mean = (features * weights).sum(1) / counts
std = (((features - mean[:, None]) ** 2 * weights).sum(1) / counts).sqrt()
pooled = {"mean": mean.numpy(), "mean_std": torch.cat([mean, std], 1).numpy()}
del weights
print("features:", tuple(features.shape), {k: v.shape[1] for k, v in pooled.items()})

## 3. Fold (ML과 같음)

ML fold 파일(`ml_1_main`이 만든 `folds.csv`)을 338명으로 제한합니다. 파일이 없으면 ML과 같은 규칙으로 fold만 새로 만듭니다 (ML 임상 데이터 준비 필요). 증강본은 원본 환자의 fold를 따라갑니다.

In [ ]:
if (
    not fold_file.is_file()
):  # ML(ml_1_main)을 아직 안 돌렸으면 ML과 같은 규칙(5-fold x 3, seed 2026)으로 fold 파일만 만듦
    from Members.mingyu.model.ML.kfold import Settings, load_cohort, save_or_check_folds

    save_or_check_folds(load_cohort(Settings()), Settings())
folds = pd.read_csv(fold_file, dtype={"patient_id": str})
folds = folds[folds.patient_id.isin(dev_ids)].reset_index(drop=True)
assert folds.groupby("repeat").patient_id.nunique().eq(len(dev_ids)).all(), (
    "ML fold에 없는 개발 환자가 있음"
)
assert folds.groupby("repeat").size().eq(len(dev_ids)).all()

label = dict(zip(cohort.patient_id, cohort.y_true, strict=False))
(
    folds.assign(scd=folds.patient_id.map(label))
    .groupby(["fold", "repeat"])
    .agg(eval_patients=("patient_id", "size"), eval_scd=("scd", "sum"))
    .unstack("repeat")
)

## 4. 학습 함수

- `split_rows`: 평가 = fold의 원본 환자. 학습 = 나머지 원본 + 그 환자들의 증강본
- `inner_split`: MIL best epoch용. 학습 원본의 20%(SCD 비율 유지)를 떼고, 그 환자들의 증강본은 학습에서도 뺌

In [ ]:
def split_rows(eval_ids):
    held = set(eval_ids)
    train = np.flatnonzero(cohort.dev & ~cohort.base_id.isin(held) & cohort.has_z)
    evaluate = np.flatnonzero(cohort.patient_id.isin(held))
    return train, evaluate


def inner_split(train_idx, seed):
    rows = cohort.iloc[train_idx]
    originals = rows[~rows.augmented]
    splitter = StratifiedShuffleSplit(
        1, test_size=settings["inner_val_fraction"], random_state=seed
    )
    _, pick = next(splitter.split(originals, originals.y_true))
    inner = set(originals.base_id.iloc[pick])
    fit = train_idx[~rows.base_id.isin(inner).to_numpy()]
    val = train_idx[(rows.base_id.isin(inner) & ~rows.augmented).to_numpy()]
    return fit, val


def fit_mil(config, train_idx, seed):
    fit, val = inner_split(train_idx, seed)
    fit_t, val_t = torch.from_numpy(fit), torch.from_numpy(val)
    mp.seed_everything(seed)
    head = mp.MILClassifierHead(features.shape[-1], config["hidden_dim"], config["dropout"])
    best, history = mp.train_head(
        head,
        (features[fit_t], mask[fit_t]),
        y[fit],
        (features[val_t], mask[val_t]),
        y[val],
        {**settings, **config, "seed": seed},
        device,
    )
    positives = int(y[fit].sum())
    history = (
        history.assign(  # PPT 특이사항용: 학습 파일 수와 실제 pos_weight (train_head와 같은 식)
            n_fit=len(fit),
            n_fit_scd=positives,
            n_inner_val=len(val),
            pos_weight=(len(fit) - positives) / positives,
        )
    )
    return head, best, history


def fit_probe(params, train_idx, seed):
    model = make_pipeline(
        StandardScaler(),
        PCA(params["pca"], random_state=seed),
        LogisticRegression(C=params["C"], class_weight="balanced", max_iter=5000),
    )
    return model.fit(pooled[params["pooling"]][train_idx], y[train_idx])


def fit_predict(kind, params, train_idx, predict_idx, seed):
    """Returns probability, best epoch, loss 기록.
    MIL: epoch별 train_loss / val_loss (val = 학습 fold 안에서 뗀 원본 20%).
    probe: 한 줄, train_loss (학습 파일 가중 log loss). 평가 fold loss는 교차검증 셀에서 붙임."""
    if kind == "mil":
        head, best, history = fit_mil(params, train_idx, seed)
        index = torch.from_numpy(predict_idx)
        return mp.predict(head, features[index], mask[index], device), best["epoch"], history
    model = fit_probe(params, train_idx, seed)
    train_p = np.clip(
        model.predict_proba(pooled[params["pooling"]][train_idx])[:, 1], 1e-7, 1 - 1e-7
    )
    train_y = y[train_idx]
    history = pd.DataFrame(
        [
            {
                **params,
                "n_fit": len(train_idx),
                "n_fit_scd": int(train_y.sum()),
                "train_loss": log_loss(
                    train_y, train_p, sample_weight=compute_sample_weight("balanced", train_y)
                ),
            }
        ]
    )
    return model.predict_proba(pooled[params["pooling"]][predict_idx])[:, 1], np.nan, history


methods = {name: ("mil", config) for name, config in mil_configs.items()}
for pooling, pca, C in itertools.product(probe_grid["pooling"], probe_grid["pca"], probe_grid["C"]):
    methods[f"Probe_{pooling}_pca{pca}_C{C:g}"] = (
        "probe",
        {"pooling": pooling, "pca": pca, "C": C},
    )
print(len(methods), "개 후보:", list(methods))

## 5. 교차검증 실행

후보마다 3회 × 5-fold = 15번 학습합니다. 후보 하나가 끝날 때마다 `oof/<후보>.csv`로 저장되고, 다시 실행하면 저장된 후보는 건너뜁니다. 저장된 설정과 지금 설정이 다르면 멈춥니다 (`run_tag`를 새로 정하세요).

In [ ]:
record = {
    "settings": settings,
    "mil_configs": mil_configs,
    "probe_grid": probe_grid,
    "threshold": THRESHOLD,
    "fold_file": str(fold_file.relative_to(repo)),
    "split_path": raw["split_path"],
    "dev_patients": int(len(dev_ids)),
    "dev_scd": int(cohort.y_true[cohort.patient_id.isin(dev_ids)].sum()),
}
record = json.loads(json.dumps(record, default=str))
oof_dir = result_dir / "oof"
oof_dir.mkdir(parents=True, exist_ok=True)
curve_dir = (
    result_dir / "loss_curve"
)  # 후보별 fold loss 기록 (PPT Loss 그래프용). MIL: epoch별, probe: fold별 한 줄
curve_dir.mkdir(exist_ok=True)
record_path = result_dir / "settings.json"
if record_path.exists() and json.loads(record_path.read_text(encoding="utf-8")) != record:
    raise ValueError(f"{result_dir}의 저장된 설정과 다릅니다. run_tag를 새로 정하세요")
record_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")

oof = {}
for name, (kind, params) in methods.items():
    path = oof_dir / f"{name}.csv"
    if path.is_file():
        oof[name] = pd.read_csv(path, dtype={"patient_id": str})
        continue
    parts, curves = [], []
    for (repeat, fold), eval_ids in folds.groupby(["repeat", "fold"]).patient_id:
        train_idx, eval_idx = split_rows(eval_ids)
        seed = int(settings["seed"] + 100 * repeat + fold)
        probability, epoch, history = fit_predict(kind, params, train_idx, eval_idx, seed)
        if kind == "probe":  # probe의 validation loss = 평가 fold의 가중 log loss
            p_eval, y_eval = np.clip(probability, 1e-7, 1 - 1e-7), y[eval_idx]
            history["val_loss"] = log_loss(
                y_eval, p_eval, sample_weight=compute_sample_weight("balanced", y_eval)
            )
        curves.append(history.assign(repeat=repeat, fold=fold))
        parts.append(
            pd.DataFrame(
                {
                    "patient_id": cohort.patient_id.to_numpy()[eval_idx],
                    "repeat": repeat,
                    "fold": fold,
                    "y_true": y[eval_idx],
                    "p_positive": probability,
                    "best_epoch": epoch,
                }
            )
        )
    oof[name] = pd.concat(parts, ignore_index=True)
    pd.concat(curves, ignore_index=True).to_csv(curve_dir / f"{name}.csv", index=False)
    oof[name].to_csv(path, index=False)
    print(f"{name}: OOF AUROC {roc_auc_score(oof[name].y_true, oof[name].p_positive):.3f}")

## 6. 결과 비교

반복마다 OOF 점수(338명)로 지표를 계산해 반복 평균 ± 표준편차로 봅니다. 가중 log loss는 ML과 같은 `balanced` 가중치, log loss·Brier는 가중치 없음. sensitivity·specificity는 threshold 0.5.

In [ ]:
def scores(frame):
    target = frame.y_true.to_numpy()
    probability = np.clip(frame.p_positive.to_numpy(), 1e-7, 1 - 1e-7)
    predicted = probability >= THRESHOLD
    return {
        "auroc": roc_auc_score(target, probability),
        "ap": average_precision_score(target, probability),
        "log_loss_balanced": log_loss(
            target, probability, sample_weight=compute_sample_weight("balanced", target)
        ),
        "log_loss": log_loss(target, probability),
        "brier": brier_score_loss(target, probability),
        "sensitivity": predicted[target == 1].mean(),
        "specificity": (~predicted[target == 0]).mean(),
    }


def summarize(frame):
    per_repeat = pd.DataFrame([scores(part) for _, part in frame.groupby("repeat")])
    out = per_repeat.mean()
    for column in ("auroc", "ap", "log_loss_balanced"):
        out[f"{column}_sd"] = per_repeat[column].std()
    return out


summary = pd.DataFrame({name: summarize(frame) for name, frame in oof.items()}).T
summary.insert(0, "kind", [methods[name][0] for name in summary.index])
summary["best_epoch_median"] = [frame.best_epoch.median() for frame in oof.values()]
summary = summary.sort_values("log_loss_balanced")
columns = [
    "kind",
    "auroc",
    "auroc_sd",
    "ap",
    "ap_sd",
    "log_loss_balanced",
    "log_loss_balanced_sd",
    "log_loss",
    "brier",
    "sensitivity",
    "specificity",
    "best_epoch_median",
]
summary.to_csv(result_dir / "cv_summary.csv")
print(f"기준선: AUROC 0.5, AP {y[cohort.patient_id.isin(dev_ids)].mean():.3f} (SCD 비율)")
summary[columns].round(3)

### 후보 종류별 1등 + ML과 같은 338명에서 비교

ML은 `ml_oof_file`의 OOF(같은 fold, 338명만 사용)입니다. 다른 ML 모델과 비교하려면 경로만 바꾸세요.

In [ ]:
best_per_kind = summary.groupby("kind").head(1)
rows = {name: summary.loc[name, columns[1:-1]] for name in best_per_kind.index}
if ml_oof_file.is_file():
    ml_oof = pd.read_csv(ml_oof_file, dtype={"patient_id": str})
    ml_oof = ml_oof[ml_oof.patient_id.isin(dev_ids)]  # ML은 356명으로 학습, 비교는 같은 338명에서
    assert set(ml_oof.patient_id) == set(dev_ids), "ML OOF에 없는 개발 환자가 있음"
    rows[f"ML: {ml_oof_file.parent.name}"] = summarize(ml_oof)[columns[1:-1]]
pd.DataFrame(rows).T.round(3)

## 7. 최종 모델 + test (후보 선택 후 `run_final_test = True`)

선택한 후보를 개발 338명 전체(증강 포함)로 다시 학습하고 test 85명을 한 번 평가합니다. MIL은 best epoch을 고르려고 개발 원본 20%를 떼어 둡니다 (교차검증과 같은 규칙).

In [ ]:
chosen = final_method or summary.index[0]
print("선택:", chosen, methods[chosen])
if run_final_test:
    kind, params = methods[chosen]
    train_idx = np.flatnonzero(cohort.dev)
    test_idx = np.flatnonzero(cohort.patient_id.isin(test_ids))
    probability, epoch, history = fit_predict(kind, params, train_idx, test_idx, settings["seed"])
    history.to_csv(result_dir / "test_loss_curve.csv", index=False)
    test_metrics = binary_metrics(y[test_idx], probability, THRESHOLD)
    # test AUROC 95% CI: ML final.py와 같은 bootstrap (seed 0, 양쪽 라벨이 있는 표본 2000개, 백분위수)
    y_test = y[test_idx]
    rng = np.random.default_rng(0)
    boots = [
        i
        for i in (rng.integers(0, len(y_test), len(y_test)) for _ in range(3000))
        if 0 < y_test[i].sum() < len(i)
    ][:2000]
    test_metrics["auroc_ci"] = [
        round(float(v), 3)
        for v in np.percentile(
            [roc_auc_score(y_test[i], probability[i]) for i in boots], [2.5, 97.5]
        )
    ]
    pd.DataFrame(
        {
            "patient_id": cohort.patient_id.to_numpy()[test_idx],
            "y_true": y[test_idx],
            "p_positive": probability,
            "y_pred": (probability >= THRESHOLD).astype(int),
            "model": chosen,
        }
    ).to_csv(result_dir / "test_predictions.csv", index=False)
    train_info = {  # PPT 특이사항용
        "train_files": int(len(train_idx)),
        "train_scd_files": int(y[train_idx].sum()),
        "test_patients": int(len(test_idx)),
        "test_scd": int(y_test.sum()),
        "epochs_run": int(len(history)) if kind == "mil" else None,
        "pos_weight": float(history.pos_weight.iloc[0]) if kind == "mil" else None,
        "n_fit": int(history.n_fit.iloc[0]),
    }
    mp.save_json(
        result_dir / "test_metrics.json",
        {
            "method": chosen,
            "params": params,
            "best_epoch": epoch,
            "train": train_info,
            "test": test_metrics,
        },
    )
    display(pd.Series(test_metrics))
else:
    print("test 건너뜀")